# Multimodel 2 — MobileNetV2 (Image) + BERT (Text)
## Safe Pharmacy — Multimodal Drug Classification

**Pipeline:**
- **Phase 1** — IMAGE branch (MobileNetV2 → 256-dim L2-norm feature vector): Frozen → fine-tune last 30 layers  
- **Phase 2** — TEXT branch (BERT → 256-dim L2-norm feature vector): Frozen → fine-tune last 2 encoder blocks  
- **Phase 3** — FUSION MLP (img_feat(256) + txt_feat(256) → 512 → classes)  
- **Save** → `saved_models_final/multimodel2/`

**Text Input Strategy (anti-leakage):**  
Training: extract pseudo drug name from image filename (e.g. `metformin_001.jpg` → `metformin`)  
Inference: user supplies actual drug name (e.g. `Metformin 500mg`)

## Cell 1 — Imports & Environment Setup

In [1]:
!pip install -q tensorflow==2.15.0 transformers==4.38.2 scikit-learn seaborn

ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.0 (from versions: none)
ERROR: No matching distribution found for tensorflow==2.15.0


In [2]:
!pip install -q transformers==4.49.0 tf-keras

In [3]:
import sys
!{sys.executable} -m pip install numpy matplotlib seaborn tensorflow scikit-learn transformers==4.49.0 tf-keras


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install matplotlib seaborn  


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
sys
!{sys.executable} -m pip install tensorflow


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
sys
!{sys.executable} -m pip install scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
sys
!{sys.executable} -m pip install transformers==4.49.0 tf-keras


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import re
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # REQUIRED for TF 2.18+

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tf_keras import layers, regularizers
from tf_keras.models import Model
Input = layers.Input
from tf_keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tf_keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# ✅ Use the stable public import instead of the internal path
from transformers import AutoTokenizer, TFBertModel

# Set seeds
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"transformers       : {__import__('transformers').__version__}")
print(f"GPU available      : {tf.config.list_physical_devices('GPU')}")


TensorFlow version : 2.21.0
transformers       : 4.49.0
GPU available      : []


## Cell 2 — Paths & Hyper-parameters

In [9]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf

# ── Paths (relative to notebook location) ───────────────────────────────────
BASE_DIR = Path.cwd()                          # folder where multimodel2.ipynb is
DATA_DIR = BASE_DIR / "dataset" / "processed"  # dataset/processed/train|val|test
SAVE_DIR = BASE_DIR / "saved_models_final" / "multimodel2"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"SAVE_DIR : {SAVE_DIR}")

# ── Hyper-parameters ──────────────────────────────────────────────────────────
IMG_SIZE        = (224, 224)
BATCH_SIZE      = 16
EPOCHS_FROZEN   = 15
EPOCHS_FINETUNE = 20
LR_FROZEN       = 1e-3
LR_FINETUNE     = 5e-5
LABEL_SMOOTHING = 0.05
FEATURE_DIM     = 256
MAX_TEXT_LEN    = 32
RANDOM_SEED     = 42

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Import your local module ──────────────────────────────────────────────────
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from split_and_preprocess import get_generators

# Fallback in case get_class_weights is not exported from the module
def get_class_weights(train_gen):
    from collections import Counter
    counter = Counter(train_gen.classes)
    max_count = max(counter.values())
    return {cls: max_count / count for cls, count in counter.items()}

# ── Create generators ─────────────────────────────────────────────────────────
train_gen, val_gen, test_gen = get_generators(backbone="mobilenet")

class_names   = list(train_gen.class_indices.keys())
num_classes   = len(class_names)
class_weights = get_class_weights(train_gen)

print(f"\nNumber of classes : {num_classes}")
print(f"Classes           : {class_names}")
print(f"Train samples     : {train_gen.samples}")
print(f"Val   samples     : {val_gen.samples}")
print(f"Test  samples     : {test_gen.samples}")

# Save label encoder
np.save(str(SAVE_DIR / "label_encoder.npy"), np.array(class_names))
print(f"\n✓ Label encoder saved → {SAVE_DIR}/label_encoder.npy")

BASE_DIR : c:\Users\hp\Documents\GitHub\deep_learning_project_final
DATA_DIR : c:\Users\hp\Documents\GitHub\deep_learning_project_final\dataset\processed
SAVE_DIR : c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2
Found 1407 images belonging to 11 classes.
Found 401 images belonging to 11 classes.
Found 214 images belonging to 11 classes.

Number of classes : 11
Classes           : ['analgesics_pain_fever', 'cardiovascular_blood', 'dermatology', 'diabetes_endocrine', 'eye_ear_nose_preparations', 'hormones_oncology', 'infections_immunity', 'neuro_musculo', 'respiratory_digestive', 'steroids_topicals', 'vitamins_supplements']
Train samples     : 1407
Val   samples     : 401
Test  samples     : 214

✓ Label encoder saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2/label_encoder.npy


## Cell 3 — Class Definitions (23 Classes)

In [10]:
# ── 11-class descriptions (matches actual processed dataset) ──────────────────
CLASS_INFO = {
    "analgesics_pain_fever":       "Pain relief and fever reduction.",
    "cardiovascular_blood":        "Heart and blood-pressure medicines.",
    "dermatology":                 "Skin conditions — acne, eczema, psoriasis.",
    "diabetes_endocrine":          "Blood sugar and endocrine disorders.",
    "eye_ear_nose_preparations":   "Eyes, ears, nose treatments.",
    "hormones_oncology":           "Hormones, oncology, cancer therapy.",
    "infections_immunity":         "Infections and immunity boosters.",
    "neuro_musculo":               "Neurology, psychiatry, musculoskeletal.",
    "respiratory_digestive":       "Respiratory and digestive treatments.",
    "steroids_topicals":           "Topical anti-inflammatory steroids.",
    "vitamins_supplements":        "Vitamins and mineral supplements.",
}

print(f"Total classes defined: {len(CLASS_INFO)}")
for k, v in CLASS_INFO.items():
    print(f"  {k:<35} — {v}")

Total classes defined: 11
  analgesics_pain_fever               — Pain relief and fever reduction.
  cardiovascular_blood                — Heart and blood-pressure medicines.
  dermatology                         — Skin conditions — acne, eczema, psoriasis.
  diabetes_endocrine                  — Blood sugar and endocrine disorders.
  eye_ear_nose_preparations           — Eyes, ears, nose treatments.
  hormones_oncology                   — Hormones, oncology, cancer therapy.
  infections_immunity                 — Infections and immunity boosters.
  neuro_musculo                       — Neurology, psychiatry, musculoskeletal.
  respiratory_digestive               — Respiratory and digestive treatments.
  steroids_topicals                   — Topical anti-inflammatory steroids.
  vitamins_supplements                — Vitamins and mineral supplements.


## Cell 4 — Utility Functions

In [11]:
def banner(msg: str):
    """Print a section banner."""
    print("\n" + "=" * 70)
    print(f"  {msg}")
    print("=" * 70)


def filename_to_drug_name(filename: str) -> str:
    """
    Convert image filename to a pseudo drug name for BERT input.
    e.g.  "metformin_hcl_001_aug2.jpg"  →  "metformin hcl"

    Strategy:
      1. Strip extension and augmentation suffix (_aug\\d+)
      2. Strip trailing numeric index (_\\d+)
      3. Replace underscores with spaces
      4. Lowercase
    """
    name = Path(filename).stem
    name = re.sub(r"_aug\d+$", "", name)
    name = re.sub(r"_\d+$",   "", name)
    name = name.replace("_", " ").lower().strip()
    return name if name else "unknown"


def get_class_weights(train_gen) -> dict:
    """Compute balanced class weights from a Keras generator."""
    labels  = train_gen.classes
    classes = np.unique(labels)
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    return dict(enumerate(weights))


def _print_branch_accuracy_summary(title: str, train_acc: float,
                                    val_acc: float, test_acc: float,
                                    save_path: Path):
    """Print and save a formatted accuracy summary for a branch."""
    gap_tv  = abs(train_acc - val_acc)
    gap_vte = abs(val_acc   - test_acc)
    lines = [
        "=" * 55,
        f"  Accuracy Summary — {title}",
        "=" * 55,
        f"  Train accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)",
        f"  Val   accuracy : {val_acc:.4f}  ({val_acc*100:.2f}%)",
        f"  Test  accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)",
        "-" * 55,
        f"  Train-Val gap  : {gap_tv:.4f}   {'✓ OK (<0.15)' if gap_tv < 0.15 else '⚠ HIGH (>0.15)'}",
        f"  Val-Test gap   : {gap_vte:.4f}   {'✓ OK (<0.10)' if gap_vte < 0.10 else '⚠ HIGH (>0.10)'}",
        f"  Generalisation : {'✓ STABLE' if gap_tv < 0.15 and gap_vte < 0.10 else '⚠ CHECK FOR OVERFITTING'}",
        "=" * 55,
    ]
    text = "\n".join(lines)
    print("\n" + text)
    save_path.write_text(text, encoding="utf-8")
    print(f"  ✓ Accuracy summary → {save_path}")


def _plot_history(histories: list, save_path: str, title: str = ""):
    """Plot loss and accuracy curves across one or more training histories."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    offset = 0
    for hist in histories:
        n  = len(hist.history["loss"])
        ep = range(offset, offset + n)
        axes[0].plot(ep, hist.history["loss"],         label="train_loss")
        axes[0].plot(ep, hist.history["val_loss"],     label="val_loss",
                     linestyle="--")
        axes[1].plot(ep, hist.history["accuracy"],     label="train_acc")
        axes[1].plot(ep, hist.history["val_accuracy"], label="val_acc",
                     linestyle="--")
        offset += n
    axes[0].set_title("Loss");     axes[0].legend(); axes[0].set_xlabel("Epoch")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  ✓ Plot saved → {save_path}")


def _plot_confusion_matrix(y_true, y_pred, class_names: list, save_path: str):
    """Plot and save a normalised confusion matrix heatmap."""
    cm   = confusion_matrix(y_true, y_pred)
    norm = cm.astype("float") / (cm.sum(axis=1, keepdims=True) + 1e-8)
    n    = len(class_names)
    fig_sz = max(10, n)
    fig, ax = plt.subplots(figsize=(fig_sz, fig_sz - 1))
    sns.heatmap(norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 8})
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title("Confusion Matrix (Normalised) — Multimodel 2 Test Set",
                 fontsize=13)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0,  fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  ✓ Confusion matrix → {save_path}")


print("✓ Utility functions defined.")

✓ Utility functions defined.


## Cell 5 — Data Generators

In [12]:
# Add BASE_DIR to path so split_and_preprocess_final can be imported
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from split_and_preprocess import get_generators

train_gen, val_gen, test_gen = get_generators(backbone="mobilenet")

class_names   = list(train_gen.class_indices.keys())
num_classes   = len(class_names)
class_weights = get_class_weights(train_gen)

print(f"Number of classes : {num_classes}")
print(f"Classes           : {class_names}")
print(f"Train samples     : {train_gen.samples}")
print(f"Val   samples     : {val_gen.samples}")
print(f"Test  samples     : {test_gen.samples}")

Found 1407 images belonging to 11 classes.
Found 401 images belonging to 11 classes.
Found 214 images belonging to 11 classes.
Number of classes : 11
Classes           : ['analgesics_pain_fever', 'cardiovascular_blood', 'dermatology', 'diabetes_endocrine', 'eye_ear_nose_preparations', 'hormones_oncology', 'infections_immunity', 'neuro_musculo', 'respiratory_digestive', 'steroids_topicals', 'vitamins_supplements']
Train samples     : 1407
Val   samples     : 401
Test  samples     : 214


## Cell 6 — Image Branch: Model Architecture

In [13]:
def build_image_branch(num_classes: int):
    """
    Build MobileNetV2 classifier and feature extractor.
    Returns: (full_classifier_model, feature_extractor_model)
    """
    backbone = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(*IMG_SIZE, 3),
        alpha=1.0,
    )
    backbone.trainable = False

    inp  = Input(shape=(*IMG_SIZE, 3), name="image_input")
    x    = backbone(inp, training=False)
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.Dense(FEATURE_DIM, kernel_regularizer=regularizers.l2(1e-4),
                        name="image_features")(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Activation("relu")(x)
    feat = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1),
                         name="image_l2norm")(x)

    head = layers.Dropout(0.5)(feat)
    head = layers.Dense(128, activation="relu",
                        kernel_regularizer=regularizers.l2(1e-4))(head)
    head = layers.Dropout(0.3)(head)
    out  = layers.Dense(num_classes, activation="softmax",
                        name="image_softmax")(head)

    full_model        = Model(inp, out,  name="mobilenetv2_classifier")
    feature_extractor = Model(inp, feat, name="image_feature_extractor")
    return full_model, feature_extractor


# Preview model summary
_img_full, _img_feat = build_image_branch(num_classes)
_img_full.summary()
del _img_full, _img_feat  # Free memory; will rebuild in training cell
print("\n✓ Image branch architecture defined.")



Model: "mobilenetv2_classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image_input (InputLayer)    [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 image_features (Dense)      (None, 256)               327936    
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                          

## Cell 7 — Phase 1a: Train Image Branch (Frozen Backbone)

In [14]:
banner("PHASE 1a — IMAGE BRANCH  (MobileNetV2, frozen backbone)")

full_img_model, _ = build_image_branch(num_classes)

full_img_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FROZEN),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

cbs_frozen = [
    EarlyStopping(monitor="val_accuracy", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / "img_best_frozen.keras"),
                    monitor="val_accuracy", save_best_only=True, verbose=0),
    CSVLogger(str(SAVE_DIR / "img_log_frozen.csv")),
]

hist_img_frozen = full_img_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_FROZEN, class_weight=class_weights,
    callbacks=cbs_frozen, verbose=1,
)

print("\n✓ Phase 1a complete.")


  PHASE 1a — IMAGE BRANCH  (MobileNetV2, frozen backbone)
Epoch 1/15


88/88 [==============================] - 36s 344ms/step - loss: 2.4177 - accuracy: 0.1528 - val_loss: 2.3534 - val_accuracy: 0.2519 - lr: 0.0010
Epoch 2/15
88/88 [==============================] - 32s 364ms/step - loss: 2.2661 - accuracy: 0.2445 - val_loss: 2.2522 - val_accuracy: 0.2269 - lr: 0.0010
Epoch 3/15
88/88 [==============================] - 33s 369ms/step - loss: 2.1267 - accuracy: 0.2751 - val_loss: 2.1721 - val_accuracy: 0.2319 - lr: 0.0010
Epoch 4/15
88/88 [==============================] - 25s 277ms/step - loss: 1.9894 - accuracy: 0.3284 - val_loss: 2.1306 - val_accuracy: 0.2718 - lr: 0.0010
Epoch 5/15
88/88 [==============================] - 33s 375ms/step - loss: 1.8969 - accuracy: 0.3383 - val_loss: 2.0825 - val_accuracy: 0.3167 - lr: 0.0010
Epoch 6/15
88/88 [==============================] - 34s 390ms/step - loss: 1.7781 - accuracy: 0.4009 - val_loss: 1.9925 - val_accuracy: 0.3641 - lr: 0.0010
Epo

## Cell 8 — Phase 1b: Fine-tune Image Branch (Last 30 Layers)

In [15]:
banner("PHASE 1b — IMAGE BRANCH  (MobileNetV2, fine-tune last 30 layers)")

# Unfreeze last 30 layers of the backbone
backbone = full_img_model.layers[1]
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

full_img_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FINETUNE),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

cbs_ft = [
    EarlyStopping(monitor="val_accuracy", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / "img_best_finetune.keras"),
                    monitor="val_accuracy", save_best_only=True, verbose=0),
    CSVLogger(str(SAVE_DIR / "img_log_finetune.csv")),
]

hist_img_ft = full_img_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_FINETUNE, class_weight=class_weights,
    callbacks=cbs_ft, verbose=1,
)

print("\n✓ Phase 1b complete.")


  PHASE 1b — IMAGE BRANCH  (MobileNetV2, fine-tune last 30 layers)
Epoch 1/20
88/88 [==============================] - 37s 289ms/step - loss: 1.3979 - accuracy: 0.5380 - val_loss: 2.0267 - val_accuracy: 0.3890 - lr: 5.0000e-05
Epoch 2/20
88/88 [==============================] - 24s 271ms/step - loss: 1.2830 - accuracy: 0.5700 - val_loss: 1.9583 - val_accuracy: 0.4165 - lr: 5.0000e-05
Epoch 3/20
88/88 [==============================] - 24s 267ms/step - loss: 1.2377 - accuracy: 0.6098 - val_loss: 2.1017 - val_accuracy: 0.3317 - lr: 5.0000e-05
Epoch 4/20
88/88 [==============================] - 23s 260ms/step - loss: 1.1818 - accuracy: 0.6340 - val_loss: 1.9755 - val_accuracy: 0.3990 - lr: 5.0000e-05
Epoch 5/20
88/88 [==============================] - 24s 266ms/step - loss: 1.1368 - accuracy: 0.6567 - val_loss: 1.9246 - val_accuracy: 0.4115 - lr: 5.0000e-05
Epoch 6/20
88/88 [==============================] - 37s 420ms/step - loss: 1.1190 - accuracy: 0.6667 - val_loss: 1.9560 - val_accura

## Cell 9 — Evaluate & Save Image Branch

In [16]:
banner("EVALUATION — Image Branch")

print("  Evaluating image branch on all splits…")
train_loss_img, train_acc_img = full_img_model.evaluate(train_gen, verbose=0)
val_loss_img,   val_acc_img   = full_img_model.evaluate(val_gen,   verbose=0)
test_loss_img,  test_acc_img  = full_img_model.evaluate(test_gen,  verbose=0)

_print_branch_accuracy_summary(
    "Image Branch (MobileNetV2)", train_acc_img, val_acc_img, test_acc_img,
    SAVE_DIR / "image_accuracy_summary.txt",
)

# Save image feature extractor
img_extractor = tf.keras.Model(
    inputs  = full_img_model.input,
    outputs = full_img_model.get_layer("image_l2norm").output,
    name    = "image_feature_extractor",
)
img_extractor.save(str(SAVE_DIR / "image_feature_extractor.keras"))
print(f"  ✓ Image feature extractor saved → {SAVE_DIR}/image_feature_extractor.keras")

# Plot training history
_plot_history(
    [hist_img_frozen, hist_img_ft],
    str(SAVE_DIR / "img_training_history.png"),
    title="Image Branch (MobileNetV2)",
)

print("\n✓ Image branch saved successfully.")


  EVALUATION — Image Branch
  Evaluating image branch on all splits…

  Accuracy Summary — Image Branch (MobileNetV2)
  Train accuracy : 0.9026  (90.26%)
  Val   accuracy : 0.4638  (46.38%)
  Test  accuracy : 0.4486  (44.86%)
-------------------------------------------------------
  Train-Val gap  : 0.4388   ⚠ HIGH (>0.15)
  Val-Test gap   : 0.0152   ✓ OK (<0.10)
  Generalisation : ⚠ CHECK FOR OVERFITTING
  ✓ Accuracy summary → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\image_accuracy_summary.txt
  ✓ Image feature extractor saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2/image_feature_extractor.keras
  ✓ Plot saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\img_training_history.png

✓ Image branch saved successfully.


## Cell 10 — Text Branch: Dataset & Architecture

In [32]:
class TextDataset(tf.keras.utils.Sequence):

    def __init__(self, samples, num_classes, tokenizer,
                 max_len, batch_size, shuffle=True):

        self.samples = samples
        self.num_classes = num_classes
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(samples))

        if shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):

        batch_idx = self.indices[
            idx * self.batch_size:(idx + 1) * self.batch_size
        ]

        texts = [self.samples[i][0] for i in batch_idx]
        labels = [self.samples[i][1] for i in batch_idx]

        enc = self.tokenizer(
            texts,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="np",
        )

        y = tf.keras.utils.to_categorical(labels, self.num_classes)

        return (
            enc["input_ids"].astype(np.int32),
            enc["attention_mask"].astype(np.int32),
            enc["token_type_ids"].astype(np.int32),
        ), y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

def build_text_samples_from_files(data_dir: Path, class_names: list) -> dict:
    """
    Walk processed/{train,val,test}/<class>/*.jpg and build text samples.
    Text = drug name derived from filename (NOT class label).
    Returns dict: {"train": [...], "val": [...], "test": [...]}
    """
    split_samples = {"train": [], "val": [], "test": []}
    for split in ("train", "val", "test"):
        split_dir = data_dir / split
        if not split_dir.exists():
            print(f"  ⚠ {split_dir} not found — skipping")
            continue
        for cls_dir in sorted(split_dir.iterdir()):
            if not cls_dir.is_dir() or cls_dir.name not in class_names:
                continue
            idx = class_names.index(cls_dir.name)
            for img_file in cls_dir.iterdir():
                if img_file.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    drug_name = filename_to_drug_name(img_file.name)
                    split_samples[split].append((drug_name, idx))
        np.random.seed(RANDOM_SEED)
        np.random.shuffle(split_samples[split])
    return split_samples


def build_text_branch(num_classes):

    input_ids = layers.Input(
        shape=(MAX_TEXT_LEN,),
        dtype=tf.int32,
        name="input_ids"
    )

    attention_mask = layers.Input(
        shape=(MAX_TEXT_LEN,),
        dtype=tf.int32,
        name="attention_mask"
    )

    token_type_ids = layers.Input(
        shape=(MAX_TEXT_LEN,),
        dtype=tf.int32,
        name="token_type_ids"
    )

    bert = TFBertModel.from_pretrained("bert-base-uncased")
    bert.trainable = False

    # 🔥 CRITICAL FIX: legacy-safe call
    bert_outputs = bert(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
        training=False
    )

    cls_token = bert_outputs.last_hidden_state[:, 0, :]

    x = layers.Dense(
        FEATURE_DIM,
        kernel_regularizer=regularizers.l2(1e-4)
    )(cls_token)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1),
        name="text_l2norm"
    )(x)

    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = Model(
        inputs=[input_ids, attention_mask, token_type_ids],
        outputs=out,
        name="bert_classifier"
    )

    return model


def evaluate_text_branch(model, tokenizer, split_samples, num_classes, split_name, max_len):
    """Run inference on a split and return accuracy."""
    samples = split_samples[split_name]
    if not samples:
        return 0.0
    seq = TextDataset(
        samples,
        num_classes,
        tokenizer,
        max_len=max_len,      # ← pass max_len here
        shuffle=False,
        batch_size=BATCH_SIZE,
    )
    results = model.evaluate(seq, verbose=0)
    return results[1]  # accuracy


print("✓ Text branch classes and functions defined.")

✓ Text branch classes and functions defined.


## Cell 11 — Phase 2a: Train Text Branch (Frozen BERT)

In [18]:
import sys
!{sys.executable} -m pip install hf_xet huggingface_hub[hf_xet]


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import sys
!{sys.executable} -m pip install torch


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
banner("PHASE 2a — TEXT BRANCH  (BERT, frozen backbone)")

tokenizer     = AutoTokenizer.from_pretrained("bert-base-uncased") # Changed BertTokenizer to AutoTokenizer
split_samples = build_text_samples_from_files(DATA_DIR, class_names)

n_train = len(split_samples["train"])
n_val   = len(split_samples["val"])
n_test  = len(split_samples["test"])
print(f"\n  Text samples — train: {n_train}  val: {n_val}  test: {n_test}")
print("  Sample texts (first 3 train):")
for text, lbl in split_samples["train"][:3]:
    print(f"    '{text}'  →  class={class_names[lbl]}")

train_seq = TextDataset(split_samples["train"], num_classes,
                         tokenizer, MAX_TEXT_LEN, BATCH_SIZE, shuffle=True)
val_seq   = TextDataset(split_samples["val"],   num_classes,
                         tokenizer, MAX_TEXT_LEN, BATCH_SIZE, shuffle=False)

txt_model = build_text_branch(num_classes)
txt_model.summary()

txt_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FROZEN),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

cbs_txt_frozen = [
    EarlyStopping(monitor="val_accuracy", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / "txt_best_frozen.keras"),
                    monitor="val_accuracy", save_best_only=True, verbose=0),
    CSVLogger(str(SAVE_DIR / "txt_log_frozen.csv")),
]

hist_txt_frozen = txt_model.fit(
    train_seq, validation_data=val_seq,
    epochs=EPOCHS_FROZEN, class_weight=class_weights,
    callbacks=cbs_txt_frozen, verbose=1,
)

print("\n✓ Phase 2a complete.")


  PHASE 2a — TEXT BRANCH  (BERT, frozen backbone)

  Text samples — train: 1407  val: 401  test: 214
  Sample texts (first 3 train):
    'insulins'  →  class=diabetes_endocrine
    'central nervous system'  →  class=neuro_musculo
    'gastro intestinal tract'  →  class=respiratory_digestive


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "bert_classifier"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 32)]                 0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 32)]                 0         []                            
 )                                                                                                
                                                                                                  
 token_type_ids (InputLayer  [(None, 32)]                 0         []                            
 )                                                                                                
                                                                                    

## Cell 12 — Phase 2b: Fine-tune BERT (Last 2 Encoder Blocks)

In [21]:
banner("PHASE 2b — TEXT BRANCH  (BERT, fine-tune last 2 encoder blocks)")

# Find and partially unfreeze BERT layer
bert_layer = None
for layer in txt_model.layers:
    if isinstance(layer, TFBertModel):
        bert_layer = layer
        break

if bert_layer is not None:
    bert_layer.trainable = True
    # Freeze everything except last 2 transformer encoder blocks
    # and explicitly freeze pooler to suppress gradient warning
    for enc_block in bert_layer.bert.encoder.layer[:-2]:
        enc_block.trainable = False
    bert_layer.bert.pooler.trainable = False

txt_model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FINETUNE),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

cbs_txt_ft = [
    EarlyStopping(monitor="val_accuracy", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-7, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / "txt_best_finetune.keras"),
                    monitor="val_accuracy", save_best_only=True, verbose=0),
    CSVLogger(str(SAVE_DIR / "txt_log_finetune.csv")),
]

hist_txt_ft = txt_model.fit(
    train_seq, validation_data=val_seq,
    epochs=EPOCHS_FINETUNE, class_weight=class_weights,
    callbacks=cbs_txt_ft, verbose=1,
)

print("\n✓ Phase 2b complete.")


  PHASE 2b — TEXT BRANCH  (BERT, fine-tune last 2 encoder blocks)
Epoch 1/20
88/88 [==============================] - 135s 1s/step - loss: 0.4352 - accuracy: 0.9929 - val_loss: 0.3488 - val_accuracy: 1.0000 - lr: 5.0000e-05
Epoch 2/20
88/88 [==============================] - 102s 1s/step - loss: 0.3919 - accuracy: 1.0000 - val_loss: 0.3480 - val_accuracy: 1.0000 - lr: 5.0000e-05
Epoch 3/20
88/88 [==============================] - 84s 960ms/step - loss: 0.3841 - accuracy: 1.0000 - val_loss: 0.3488 - val_accuracy: 1.0000 - lr: 5.0000e-05
Epoch 4/20
88/88 [==============================] - 99s 1s/step - loss: 0.3866 - accuracy: 0.9993 - val_loss: 0.3477 - val_accuracy: 1.0000 - lr: 5.0000e-05
Epoch 5/20
88/88 [==============================] - 113s 1s/step - loss: 0.3809 - accuracy: 1.0000 - val_loss: 0.3478 - val_accuracy: 1.0000 - lr: 5.0000e-05
Epoch 6/20
88/88 [==============================] - 106s 1s/step - loss: 0.3768 - accuracy: 1.0000 - val_loss: 0.3486 - val_accuracy: 1.0000 -

## Cell 13 — Evaluate & Save Text Branch

In [33]:
banner("EVALUATION — Text Branch (BERT)")

MAX_LEN = 128

print("  Evaluating BERT branch on all splits…")
train_acc_txt = evaluate_text_branch(txt_model, tokenizer, split_samples,
                                      num_classes, "train", MAX_LEN)
val_acc_txt   = evaluate_text_branch(txt_model, tokenizer, split_samples,
                                      num_classes, "val", MAX_LEN)
test_acc_txt  = evaluate_text_branch(txt_model, tokenizer, split_samples,
                                      num_classes, "test", MAX_LEN)

_print_branch_accuracy_summary(
    "Text Branch (BERT)", train_acc_txt, val_acc_txt, test_acc_txt,
    SAVE_DIR / "text_accuracy_summary.txt",
)

# Save text feature extractor
txt_extractor = tf.keras.Model(
    inputs  = txt_model.input,
    outputs = txt_model.get_layer("text_l2norm").output,
    name    = "text_feature_extractor",
)
txt_extractor.save(str(SAVE_DIR / "text_feature_extractor"))
print(f"  ✓ Text feature extractor saved → {SAVE_DIR}/text_feature_extractor/")

# Save tokenizer
tokenizer.save_pretrained(str(SAVE_DIR / "tokenizer"))
print(f"  ✓ Tokenizer saved → {SAVE_DIR}/tokenizer/")

# Plot training history
_plot_history(
    [hist_txt_frozen, hist_txt_ft],
    str(SAVE_DIR / "txt_training_history.png"),
    title="Text Branch (BERT)",
)

print("\n✓ Text branch saved successfully.")


  EVALUATION — Text Branch (BERT)
  Evaluating BERT branch on all splits…

  Accuracy Summary — Text Branch (BERT)
  Train accuracy : 1.0000  (100.00%)
  Val   accuracy : 1.0000  (100.00%)
  Test  accuracy : 1.0000  (100.00%)
-------------------------------------------------------
  Train-Val gap  : 0.0000   ✓ OK (<0.15)
  Val-Test gap   : 0.0000   ✓ OK (<0.10)
  Generalisation : ✓ STABLE
  ✓ Accuracy summary → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\text_accuracy_summary.txt
INFO:tensorflow:Assets written to: c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\text_feature_extractor\assets


INFO:tensorflow:Assets written to: c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\text_feature_extractor\assets


  ✓ Text feature extractor saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2/text_feature_extractor/
  ✓ Tokenizer saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2/tokenizer/
  ✓ Plot saved → c:\Users\hp\Documents\GitHub\deep_learning_project_final\saved_models_final\multimodel2\txt_training_history.png

✓ Text branch saved successfully.


## Cell 14 — Fusion Model: Architecture & Feature Extraction

In [34]:
def build_fusion_model(num_classes: int) -> Model:
    """Build MLP fusion model: img_feat(256) + txt_feat(256) → classes."""
    img_input = Input(shape=(FEATURE_DIM,), name="image_features_in")
    txt_input = Input(shape=(FEATURE_DIM,), name="text_features_in")

    fused = layers.Concatenate(name="fusion_concat")([img_input, txt_input])

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4),
                     name="mlp_256")(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4),
                     name="mlp_128")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation="softmax",
                       name="fusion_output")(x)

    model = Model([img_input, txt_input], out, name="fusion_model_multimodel2")
    return model


def extract_image_feats(img_extractor: Model, gen, n_samples: int) -> tuple:
    """Extract image feature vectors from a Keras generator."""
    feats, labels = [], []
    steps = int(np.ceil(n_samples / BATCH_SIZE))
    for i, (batch_x, batch_y) in enumerate(gen):
        feats.append(img_extractor.predict(batch_x, verbose=0))
        labels.append(batch_y)
        if i + 1 >= steps:
            break
    return np.vstack(feats)[:n_samples], np.vstack(labels)[:n_samples]


def extract_text_feats(txt_extractor: Model, tokenizer,
                        samples: list) -> np.ndarray:
    """Extract BERT feature vectors from (drug_name, label) sample list."""
    all_feats = []
    for i in range(0, len(samples), BATCH_SIZE):
        batch_texts = [s[0] for s in samples[i: i + BATCH_SIZE]]
        enc = tokenizer(
            batch_texts, max_length=MAX_TEXT_LEN, padding="max_length",
            truncation=True, return_tensors="np",
        )
        f = txt_extractor.predict(
            [enc["input_ids"].astype(np.int32),
             enc["attention_mask"].astype(np.int32),
             enc["token_type_ids"].astype(np.int32)],
            verbose=0,
        )
        all_feats.append(f)
    return np.vstack(all_feats)


print("✓ Fusion model architecture and feature extraction functions defined.")

✓ Fusion model architecture and feature extraction functions defined.


## Cell 15 — Phase 3: Extract Features for Fusion

In [35]:
banner("PHASE 3 — FUSION MODEL  (image 256 + text 256 → MLP → classes)")

# Reset generators to ensure correct ordering
train_gen, val_gen, test_gen = get_generators(backbone="mobilenet")

# ── Extract image features ────────────────────────────────────────────────────
print("  Extracting image features (train)…")
train_img, train_lbl = extract_image_feats(img_extractor, train_gen,
                                            train_gen.samples)
print("  Extracting image features (val)…")
val_img,   val_lbl   = extract_image_feats(img_extractor, val_gen,
                                            val_gen.samples)
print("  Extracting image features (test)…")
test_img,  test_lbl  = extract_image_feats(img_extractor, test_gen,
                                            test_gen.samples)

# ── Build text samples aligned with generator order ───────────────────────────
def gen_order_samples(gen, class_names: list) -> list:
    """
    Rebuild (drug_name, class_idx) list in the same order as the generator.
    gen.filenames gives paths in generator order.
    """
    out = []
    for fpath in gen.filenames:
        fname    = Path(fpath).name
        cls_name = Path(fpath).parent.name
        label    = class_names.index(cls_name) if cls_name in class_names else 0
        drug     = filename_to_drug_name(fname)
        out.append((drug, label))
    return out

print("  Building text samples aligned with generator order…")
train_txt_samples = gen_order_samples(train_gen, class_names)
val_txt_samples   = gen_order_samples(val_gen,   class_names)
test_txt_samples  = gen_order_samples(test_gen,  class_names)

# ── Extract text features ─────────────────────────────────────────────────────
print("  Extracting text features (train)…")
train_txt = extract_text_feats(txt_extractor, tokenizer, train_txt_samples)
print("  Extracting text features (val)…")
val_txt   = extract_text_feats(txt_extractor, tokenizer, val_txt_samples)
print("  Extracting text features (test)…")
test_txt  = extract_text_feats(txt_extractor, tokenizer, test_txt_samples)

# ── Align lengths (safety trim) ───────────────────────────────────────────────
n_tr = min(len(train_img), len(train_txt))
n_v  = min(len(val_img),   len(val_txt))
n_te = min(len(test_img),  len(test_txt))

train_img, train_txt, train_lbl = train_img[:n_tr], train_txt[:n_tr], train_lbl[:n_tr]
val_img,   val_txt,   val_lbl   = val_img[:n_v],   val_txt[:n_v],   val_lbl[:n_v]
test_img,  test_txt,  test_lbl  = test_img[:n_te],  test_txt[:n_te],  test_lbl[:n_te]

print(f"\n  Feature shapes:")
print(f"    train_img={train_img.shape}  train_txt={train_txt.shape}  train_lbl={train_lbl.shape}")
print(f"    val_img  ={val_img.shape}    val_txt  ={val_txt.shape}    val_lbl  ={val_lbl.shape}")
print(f"    test_img ={test_img.shape}   test_txt ={test_txt.shape}   test_lbl ={test_lbl.shape}")
print("\n✓ Feature extraction complete.")


  PHASE 3 — FUSION MODEL  (image 256 + text 256 → MLP → classes)
Found 1407 images belonging to 11 classes.
Found 401 images belonging to 11 classes.
Found 214 images belonging to 11 classes.
  Extracting image features (train)…
  Extracting image features (val)…
  Extracting image features (test)…
  Building text samples aligned with generator order…
  Extracting text features (train)…
  Extracting text features (val)…
  Extracting text features (test)…

  Feature shapes:
    train_img=(1407, 256)  train_txt=(1407, 256)  train_lbl=(1407, 11)
    val_img  =(401, 256)    val_txt  =(401, 256)    val_lbl  =(401, 11)
    test_img =(214, 256)   test_txt =(214, 256)   test_lbl =(214, 11)

✓ Feature extraction complete.


## Cell 16 — Phase 3: Train Fusion Model

In [36]:
fusion = build_fusion_model(num_classes)
fusion.summary()

fusion.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

cbs_fusion = [
    EarlyStopping(monitor="val_accuracy", patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(SAVE_DIR / "fusion_best.keras"),
                    monitor="val_accuracy", save_best_only=True, verbose=0),
    CSVLogger(str(SAVE_DIR / "fusion_log.csv")),
]

hist_fusion = fusion.fit(
    [train_img, train_txt], train_lbl,
    validation_data=([val_img, val_txt], val_lbl),
    epochs=40, batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=cbs_fusion, verbose=1,
)

# Save fusion model
fusion.save(str(SAVE_DIR / "fusion_model.keras"))
print(f"  ✓ Fusion model saved → {SAVE_DIR}/fusion_model.keras")

# Plot fusion training history
_plot_history(
    [hist_fusion],
    str(SAVE_DIR / "fusion_training_history.png"),
    title="Fusion Model (Multimodel 2)",
)

print("\n✓ Fusion model training complete.")

Model: "fusion_model_multimodel2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image_features_in (InputLa  [(None, 256)]                0         []                            
 yer)                                                                                             
                                                                                                  
 text_features_in (InputLay  [(None, 256)]                0         []                            
 er)                                                                                              
                                                                                                  
 fusion_concat (Concatenate  (None, 512)                  0         ['image_features_in[0][0]',   
 )                                                                   'text_

## Cell 17 — Final Evaluation: Fusion Model on Test Set

In [37]:
banner("EVALUATION — Fusion Model Test Set")

pred_probs = fusion.predict([test_img, test_txt], verbose=0)
y_pred     = np.argmax(pred_probs, axis=1)
y_true     = np.argmax(test_lbl,   axis=1)

# Classification report
report = classification_report(y_true, y_pred,
                                target_names=class_names, digits=4)
print(report)

rp_path = SAVE_DIR / "classification_report.txt"
rp_path.write_text(report, encoding="utf-8")
print(f"  ✓ Classification report → {rp_path}")

# Confusion matrix
_plot_confusion_matrix(y_true, y_pred, class_names,
                        str(SAVE_DIR / "confusion_matrix.png"))

# Fusion accuracy summary
f_train_loss, f_train_acc = fusion.evaluate([train_img, train_txt],
                                             train_lbl, verbose=0)
f_val_loss,   f_val_acc   = fusion.evaluate([val_img,   val_txt],
                                             val_lbl,   verbose=0)
f_test_acc = accuracy_score(y_true, y_pred)

_print_branch_accuracy_summary(
    "Fusion Model (Multimodel 2)", f_train_acc, f_val_acc, f_test_acc,
    SAVE_DIR / "fusion_accuracy_summary.txt",
)

print("\n✓ Evaluation complete.")


  EVALUATION — Fusion Model Test Set
                           precision    recall  f1-score   support

    analgesics_pain_fever     0.3333    0.2308    0.2727        13
     cardiovascular_blood     0.4762    0.4762    0.4762        21
              dermatology     0.5556    0.5556    0.5556         9
       diabetes_endocrine     0.4667    0.5000    0.4828        14
eye_ear_nose_preparations     0.7500    0.8571    0.8000        14
        hormones_oncology     0.4091    0.3750    0.3913        24
      infections_immunity     0.2333    0.2692    0.2500        26
            neuro_musculo     0.4583    0.3667    0.4074        30
    respiratory_digestive     0.5455    0.5714    0.5581        42
        steroids_topicals     0.2941    0.3571    0.3226        14
     vitamins_supplements     0.4286    0.4286    0.4286         7

                 accuracy                         0.4486       214
                macro avg     0.4501    0.4534    0.4496       214
             weighted 

## Cell 18 — Summary of Saved Outputs

In [ ]:
banner("multimodel2 — DONE")

print(f"\n  All outputs saved in: {SAVE_DIR}")
print("""
  Saved files:
    image_feature_extractor.keras   — MobileNetV2 → 256-dim (L2-norm)
    text_feature_extractor/         — BERT → 256-dim (L2-norm) [SavedModel]
    tokenizer/                      — BERT tokenizer
    fusion_model.keras              — Full multimodal fusion model
    label_encoder.npy               — class name array
    image_accuracy_summary.txt      — image branch train/val/test accuracy
    text_accuracy_summary.txt       — BERT branch train/val/test accuracy
    fusion_accuracy_summary.txt     — fusion model train/val/test accuracy
    classification_report.txt       — fusion per-class test metrics
    confusion_matrix.png            — fusion test confusion matrix
    img_training_history.png        — image branch loss/accuracy curves
    txt_training_history.png        — BERT branch loss/accuracy curves
    fusion_training_history.png     — fusion model loss/accuracy curves
    img_log_frozen.csv              — image branch frozen training log
    img_log_finetune.csv            — image branch finetune training log
    txt_log_frozen.csv              — BERT branch frozen training log
    txt_log_finetune.csv            — BERT branch finetune training log
    fusion_log.csv                  — fusion model training log
""")

# List all saved files
print("  Files currently in SAVE_DIR:")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"    {f.relative_to(SAVE_DIR)}  ({size_kb:.1f} KB)")